In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load training and test data

train_df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test_df  = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

In [ ]:
# Exploratory Data Analysis

# Distribution of target variable (SalePrice)
plt.figure(figsize=(10,6))
sns.histplot(train_df['SalePrice'], bins=40, kde=True)
plt.title("Distribution of House Sale Prices", fontsize=16)
plt.xlabel("Sale Price", fontsize=14)
plt.ylabel("Frequency", fontsize=14)
plt.show()

# Also check log-transformed SalePrice (often more normal)
plt.figure(figsize=(10,6))
sns.histplot(np.log1p(train_df['SalePrice']), bins=40, kde=True)
plt.title("Distribution of log1p(SalePrice)", fontsize=16)
plt.xlabel("log1p(SalePrice)", fontsize=14)
plt.ylabel("Frequency", fontsize=14)
plt.show()

# Correlation heatmap of top numeric features with SalePrice
numeric_features = train_df.select_dtypes(include=['int64', 'float64'])
correlation = numeric_features.corr()

top_corr_features = correlation['SalePrice'].abs().sort_values(ascending=False).head(11).index
plt.figure(figsize=(10,8))
sns.heatmap(train_df[top_corr_features].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap of Top 10 Numeric Features", fontsize=16)
plt.show()

# Boxplot of SalePrice by Neighborhood
plt.figure(figsize=(14,8))
sns.boxplot(x='Neighborhood', y='SalePrice', data=train_df)
plt.title("Sale Price Distribution by Neighborhood", fontsize=16)
plt.xlabel("Neighborhood", fontsize=14)
plt.ylabel("Sale Price", fontsize=14)
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Data Cleaning & Feature Engineering 

from sklearn.impute import SimpleImputer

# Copy original data
df = train_df.copy()
df_test = test_df.copy()

# Feature Engineering Function
def add_features(df):
    # Total square footage (basement + 1st + 2nd floors)
    df['TotalSF'] = (
        df.get('TotalBsmtSF', 0).fillna(0) +
        df.get('1stFlrSF', 0).fillna(0) +
        df.get('2ndFlrSF', 0).fillna(0)
    )
    
    # House age at time of sale
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    
    # Whether house was remodeled
    df['Remodeled'] = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)
    
    # Total bathrooms (full + half + basement)
    df['TotalBathrooms'] = (
        df.get('FullBath',0).fillna(0) +
        0.5 * df.get('HalfBath',0).fillna(0) +
        df.get('BsmtFullBath',0).fillna(0) +
        0.5 * df.get('BsmtHalfBath',0).fillna(0)
    )
    
    # Combined porch area
    porch_cols = ['OpenPorchSF','EnclosedPorch','3SsnPorch','ScreenPorch']
    df['TotalPorch'] = df[porch_cols].sum(axis=1)
    
    # Treat MSSubClass as categorical (it’s really a category, not numeric)
    df['MSSubClass'] = df['MSSubClass'].astype(str)
    
    return df

# Apply feature engineering
df = add_features(df)
df_test = add_features(df_test)

# Separate target and features 
y = df['SalePrice']
X = df.drop(['SalePrice', 'Id'], axis=1)
X_test = df_test.drop(['Id'], axis=1)

# Identify column types
numeric_cols = X.select_dtypes(include=['int64','float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

# Handle Missing Values 
# Numeric → median
num_imputer = SimpleImputer(strategy='median')
X[numeric_cols] = num_imputer.fit_transform(X[numeric_cols])
X_test[numeric_cols] = num_imputer.transform(X_test[numeric_cols])

# Categorical → most frequent
cat_imputer = SimpleImputer(strategy='most_frequent')
X[categorical_cols] = cat_imputer.fit_transform(X[categorical_cols])
X_test[categorical_cols] = cat_imputer.transform(X_test[categorical_cols])

# One-hot encode categorical features
X = pd.get_dummies(X, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Align train/test to have same columns
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

print("X shape after preprocessing:", X.shape)
print("X_test shape after preprocessing:", X_test.shape)


In [ ]:
# Baseline Model Training & Evaluation (Linear Regression) 

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Train/validation split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Initialize and train Linear Regression model
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

# Predict on validation set
y_pred = lin_reg.predict(X_valid)

# Evaluation metrics
r2 = r2_score(y_valid, y_pred)
mse = mean_squared_error(y_valid, y_pred)
rmse = np.sqrt(mse)

print("Baseline Model: Linear Regression")
print(f"R^2 Score: {r2:.4f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")


In [ ]:
# Feature Engineering Improvements + Retrain Linear Regression 

# Train/validation split again
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Retrain Linear Regression with improved features
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

# Predict on validation set
y_pred = lin_reg.predict(X_valid)

# Evaluation metrics
r2 = r2_score(y_valid, y_pred)
mse = mean_squared_error(y_valid, y_pred)
rmse = np.sqrt(mse)

print("After Feature Engineering (Linear Regression):")
print(f"R^2 Score: {r2:.4f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")


In [ ]:
# Test Multiple Models (Random Forest, Gradient Boosting, XGBoost) 

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb

# Re-split (same as before for fair comparison)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Dictionary to hold models
models = {
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=200, random_state=42, n_jobs=-1)
}

results = {}

# Train and evaluate each model
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)

    r2 = r2_score(y_valid, y_pred)
    mse = mean_squared_error(y_valid, y_pred)
    rmse = np.sqrt(mse)

    results[name] = {"R2": r2, "MSE": mse, "RMSE": rmse}
    print(f"{name}:")
    print(f"  R^2 Score: {r2:.4f}")
    print(f"  MSE: {mse:.2f}")
    print(f"  RMSE: {rmse:.2f}")
    print("-"*40)

# Compare all models in a DataFrame
results_df = pd.DataFrame(results).T
print("\nModel Comparison:")
print(results_df)


In [ ]:
# Hyperparameter Tuning (Gradient Boosting) 

from sklearn.model_selection import GridSearchCV

# Define parameter grid for Gradient Boosting
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5],
    'min_samples_split': [2, 5, 10]
}

gb_model = GradientBoostingRegressor(random_state=42)

grid_search = GridSearchCV(
    estimator=gb_model,
    param_grid=param_grid,
    cv=3,              # 3-fold cross-validation
    scoring='r2',      # optimize for R²
    n_jobs=-1,
    verbose=2
)

# Run grid search
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV R² Score:", grid_search.best_score_)

# Refit with best model
best_gb = grid_search.best_estimator_

# Evaluate on validation set
y_pred = best_gb.predict(X_valid)
r2 = r2_score(y_valid, y_pred)
mse = mean_squared_error(y_valid, y_pred)
rmse = np.sqrt(mse)

print("\nTuned Gradient Boosting Performance:")
print(f"R^2 Score: {r2:.4f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")


In [ ]:
# Feature Importance Visualization

# Get feature importances from the tuned Gradient Boosting model
importances = best_gb.feature_importances_
feature_names = X_train.columns

# Create a DataFrame for sorting
feat_importances = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Plot Top 10
plt.figure(figsize=(10,6))
sns.barplot(x='Importance', y='Feature', data=feat_importances.head(10), palette="viridis")
plt.title("Top 10 Most Important Features (Gradient Boosting)", fontsize=16)
plt.xlabel("Feature Importance", fontsize=14)
plt.ylabel("Feature", fontsize=14)
plt.show()

# Display full table (optional)
feat_importances.head(20)


In [ ]:
# Generate Predictions + Kaggle Submission File

# Retrain best model on full training data
best_gb.fit(X, y)

# Copy test set
X_test = test_df.copy()

# --- Add engineered features (same as training set) ---
X_test['TotalSF'] = X_test['TotalBsmtSF'] + X_test['1stFlrSF'] + X_test['2ndFlrSF']
X_test['HouseAge'] = X_test['YrSold'] - X_test['YearBuilt']
X_test['Remodeled'] = (X_test['YearRemodAdd'] != X_test['YearBuilt']).astype(int)
X_test['TotalBathrooms'] = (
    X_test['FullBath'] 
    + 0.5*X_test['HalfBath'] 
    + X_test['BsmtFullBath'] 
    + 0.5*X_test['BsmtHalfBath']
)
X_test['TotalPorch'] = (
    X_test['OpenPorchSF'] 
    + X_test['EnclosedPorch'] 
    + X_test['3SsnPorch'] 
    + X_test['ScreenPorch']
)

# --- Handle missing values ---
X_test[numeric_cols] = num_imputer.transform(X_test[numeric_cols])
X_test[categorical_cols] = cat_imputer.transform(X_test[categorical_cols])

# --- One-hot encode (align with train columns) ---
X_test_encoded = pd.get_dummies(X_test)
X_test_encoded = X_test_encoded.reindex(columns=X_train.columns, fill_value=0)

# --- Predict SalePrice ---
test_preds = best_gb.predict(X_test_encoded)

# --- Create submission file ---
submission = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': test_preds
})

submission.to_csv("submission.csv", index=False)
print("submission.csv created successfully!")
